# 05 — Vector Database & Knowledge Base Indexing

**Phase 6:** Qdrant setup · HNSW collection · chunk indexing · retrieval testing

### Cloud setup (recommended)
1. Create free account → [cloud.qdrant.io](https://cloud.qdrant.io)
2. Create cluster → copy URL + API key
3. Add to `.env` in project root:
```
QDRANT_URL=https://xxxx.us-east4-0.gcp.cloud.qdrant.io
QDRANT_API_KEY=your_api_key_here
```
Without these, the notebook uses a local on-disk Qdrant instance automatically.

In [ ]:
import sys, os, json, time
from pathlib import Path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from src.utils.vector_db  import VectorDBManager
from src.utils.embedder   import Embedder

plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})

db  = VectorDBManager()
emb = Embedder()
print(db)
print(f'Mode: {"☁ Cloud" if db.is_cloud else "💻 Local (set QDRANT_URL for cloud)"}')

## 1. Qdrant Setup & HNSW Configuration

In [ ]:
import yaml
cfg = yaml.safe_load((ROOT / 'configs' / 'qdrant_config.yaml').read_text())

print('Collection config:')
print(f"  Name         : {cfg['collection']['name']}")
print(f"  Vector size  : {cfg['collection']['vector_size']} (all-MiniLM-L6-v2)")
print(f"  Distance     : {cfg['collection']['distance']}")
print()
print('HNSW parameters:')
print(f"  m            : {cfg['hnsw']['m']}  (bi-directional links — higher=better recall, more RAM)")
print(f"  ef_construct : {cfg['hnsw']['ef_construct']}  (build-time candidates — higher=better quality)")
print(f"  ef (search)  : {cfg['search']['ef']}  (runtime candidates — higher=better recall)")
print()
print('Why HNSW?')
print('  • Approximate nearest-neighbour: O(log N) vs O(N) brute-force')
print('  • m=16, ef=200 gives ~98% recall at <5ms for 100k vectors')
print('  • COSINE distance correct for L2-normalised embeddings')

In [ ]:
# Create collection (idempotent — skips if exists)
created = db.create_collection(recreate=False)
info    = db.collection_info()
print(f'Collection created : {created}')
print(f'Collection info    : {info}')

## 2. Load & Inspect Knowledge Base

In [ ]:
KB_PATH = ROOT / 'data' / 'processed' / 'knowledge_base_combined.json'

if KB_PATH.exists():
    with open(KB_PATH) as f:
        chunks = json.load(f)
    print(f'Loaded {len(chunks):,} chunks from knowledge_base_combined.json')
else:
    # Demo chunks for notebook exploration
    print('⚠  knowledge_base_combined.json not found — using 20 demo chunks')
    print('   Run scripts/00_prepare_data.py (Phase 1) to generate the full KB')
    items = [
        ("Anxiety disorders involve persistent and excessive worry. CBT and medication are first-line treatments.",  "counseling", "Anxiety"),
        ("Depression is characterised by persistent low mood, loss of interest, fatigue and hopelessness.",          "counseling", "Depression"),
        ("Panic attacks involve sudden intense fear, racing heartbeat and shortness of breath.",                     "counseling", "Panic"),
        ("Cognitive Behavioral Therapy helps restructure negative thoughts and develop coping strategies.",           "pdf",        "CBT"),
        ("Insomnia and sleep disorders frequently accompany anxiety and depression. Sleep hygiene helps.",            "counseling", "Sleep"),
        ("Mindfulness meditation reduces stress, improves focus and promotes emotional regulation.",                  "pdf",        "Mindfulness"),
        ("Workplace stress and occupational burnout are common triggers for anxiety and depression.",                 "pdf",        "Work"),
        ("Relationship difficulties and conflict are major stressors linked to depression.",                         "counseling", "Relationships"),
        ("Trauma and PTSD cause flashbacks, hypervigilance and emotional numbing.",                                  "pdf",        "Trauma"),
        ("Social anxiety disorder involves intense fear of social situations and judgement.",                        "counseling", "Social Anxiety"),
        ("Grief and bereavement involve sadness, denial, anger and acceptance.",                                     "counseling", "Grief"),
        ("Exercise and physical activity significantly reduce symptoms of anxiety and depression.",                   "pdf",        "Exercise"),
        ("Breathing exercises like diaphragmatic breathing reduce acute anxiety and panic.",                         "pdf",        "Breathing"),
        ("Building social support networks protects against depression and loneliness.",                             "pdf",        "Social Support"),
        ("Medication such as SSRIs is used alongside therapy for moderate to severe depression.",                    "counseling", "Medication"),
        ("Boundaries in relationships protect emotional health and reduce resentment.",                              "pdf",        "Boundaries"),
        ("Crisis intervention and safety planning are essential for suicidal thoughts.",                             "counseling", "Crisis"),
        ("ADHD affects attention, impulsivity and emotional regulation across the lifespan.",                        "counseling", "ADHD"),
        ("Eating disorders involve distorted body image requiring specialist care.",                                 "counseling", "Eating Disorders"),
        ("Self-compassion and self-care practices are essential for mental health recovery.",                        "pdf",        "Self-care"),
    ]
    chunks = [{"id":f"demo_{i:03d}","text":t,"source":s,"source_type":s,"section":sec,"tokens":len(t.split())}
              for i,(t,s,sec) in enumerate(items)]

# Stats
import pandas as pd
df = pd.DataFrame(chunks)
print(f'\nShape      : {df.shape}')
print(f'Columns    : {list(df.columns)}')
print(f'\nSource types:')
print(df['source_type'].value_counts().to_string())

In [ ]:
# Token distribution
df['tokens_est'] = df['text'].str.split().apply(len)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['tokens_est'].hist(bins=30, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Token distribution per chunk')
axes[0].set_xlabel('Tokens (estimated)')

src_counts = df['source_type'].value_counts()
axes[1].bar(src_counts.index, src_counts.values, color=['#3498db','#2ecc71','#e74c3c'][:len(src_counts)])
axes[1].set_title('Chunks by source type')

plt.suptitle('Knowledge Base Statistics', fontsize=13)
plt.tight_layout(); plt.show()

print(df['tokens_est'].describe().round(1).to_dict())

## 3. Embed & Index Chunks

In [ ]:
print(f'Embedding {len(chunks):,} chunks …')
t0   = time.time()
texts = [c.get('text','') for c in chunks]
vecs  = emb.embed_batch(texts, batch_size=64, show_progress=True)
elapsed = time.time() - t0
print(f'\nEmbedding complete:')
print(f'  Shape   : {vecs.shape}')
print(f'  Dtype   : {vecs.dtype}')
print(f'  Time    : {elapsed:.1f}s  ({len(chunks)/elapsed:.0f} chunks/s)')
print(f'  Norms   : min={np.linalg.norm(vecs,axis=1).min():.4f}  max={np.linalg.norm(vecs,axis=1).max():.4f}  (should be ~1.0)')

In [ ]:
# Index (recreate=True to ensure clean state)
db.create_collection(recreate=True)
print('Indexing …')
t0      = time.time()
summary = db.index_chunks(chunks, vecs, show_progress=True)
print(f'\nIndexing summary:')
for k, v in summary.items():
    print(f'  {k:<12}: {v}')

count = db.verify_count()
print(f'\nVerification: {count:,} points in Qdrant  {"✅" if count == len(chunks) else "⚠"}')

## 4. Retrieval Testing

In [ ]:
TEST_QUERIES = [
    ('I feel anxious about work',          'Work / Anxiety'),
    ('How do I manage depression?',         'Depression'),
    ("I can't sleep, sleep issues",         'Sleep / Insomnia'),
    ('panic attack symptoms and treatment', 'Panic'),
    ('I feel lonely and isolated',          'Social Support'),
    ('I had a traumatic experience',        'Trauma / PTSD'),
]

query_times = []
for query, category in TEST_QUERIES:
    t0      = time.time()
    results = db.search_by_text(query, emb, limit=3, score_threshold=0.0)
    ms      = (time.time()-t0)*1000
    query_times.append(ms)
    print(f'\n  [{category}] "{query}"  ({ms:.0f}ms)')
    for r in results:
        print(f"     score={r['score']:.3f}  [{r['source_type']}]  {r['text'][:70]}")

print(f'\nQuery time: avg={sum(query_times)/len(query_times):.0f}ms  '
      f'max={max(query_times):.0f}ms  (< 500ms target {"✅" if max(query_times)<500 else "❌"})')

In [ ]:
# Verify metadata is preserved in payloads
results = db.search_by_text('anxiety coping strategies', emb, limit=3, score_threshold=0.0)
print('Payload fields returned:')
if results:
    for key, val in results[0].items():
        print(f'  {key:<15}: {repr(val)[:60]}')
    print(f'\n✅ All metadata fields present: {list(results[0].keys())}')
else:
    print('No results returned')

## 5. Score Threshold & Limit Analysis

In [ ]:
# How threshold affects number of returned results
query = 'anxiety depression treatment therapy'
vec   = emb.embed_text(query)
all_results = db.search(vec, limit=len(chunks), score_threshold=0.0)
scores = [r['score'] for r in all_results]

print(f'All scores for: "{query}"')
print(f'  Count  : {len(scores)}')
print(f'  Min    : {min(scores):.4f}')
print(f'  Max    : {max(scores):.4f}')
print(f'  Mean   : {sum(scores)/len(scores):.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(scores, bins=20, color='steelblue', edgecolor='white')
axes[0].axvline(0.45, color='red', linestyle='--', label='default threshold 0.45')
axes[0].set_title(f'Score distribution for\n"{query[:40]}"')
axes[0].set_xlabel('Cosine similarity'); axes[0].legend()

# Threshold sweep
thresholds = [0.0, 0.3, 0.4, 0.45, 0.5, 0.6, 0.7, 0.8]
counts = [sum(1 for s in scores if s >= t) for t in thresholds]
axes[1].plot(thresholds, counts, 'o-', color='#e74c3c')
axes[1].axvline(0.45, color='grey', linestyle='--', label='config threshold')
axes[1].set_title('Results returned vs threshold')
axes[1].set_xlabel('Score threshold'); axes[1].set_ylabel('Chunks returned')
axes[1].legend()

plt.suptitle('Retrieval Analysis', fontsize=13)
plt.tight_layout(); plt.show()

## 6. Cloud Setup Instructions

In [ ]:
print("""
Qdrant Cloud Setup (free tier)
══════════════════════════════════════════════════════════════════
1.  Go to https://cloud.qdrant.io  →  Sign up (free)
2.  Click  "Create cluster"
    Name  : mental-health-bot
    Plan  : Free  (1 node, 0.5 vCPU, 1 GB RAM, 4 GB storage)
    Region: pick closest to you
3.  Copy:
    • Cluster URL   e.g. https://abc123.us-east4-0.gcp.cloud.qdrant.io
    • API Key       e.g. eyJ...
4.  Create  .env  in project root:
      QDRANT_URL=https://abc123.us-east4-0.gcp.cloud.qdrant.io
      QDRANT_API_KEY=eyJ...
5.  Re-run  scripts/03_setup_rag.py  to index the full knowledge base

HNSW Parameters Explanation
══════════════════════════════════════════════════════════════════
  m=16          Each node connects to 16 neighbours.
                Higher m → better recall, more RAM (m=16 is standard).
  ef_construct=200
                Candidates considered during index build.
                Higher → better graph quality, slower build.
  ef=200        Candidates during search.
                Higher → better recall, slower query (200 is safe).
                Qdrant recommends ef ≥ limit (we use 200 for top-5).

Expected performance (cloud, 500 chunks):
  Index time : ~10-30s
  Query time : 5-50ms
  Recall@5   : ~98%
══════════════════════════════════════════════════════════════════
""")